# YOLO 학습 실행
> **실행 전 체크**: `02_EDA_track_b.ipynb` 먼저 실행 → class_id 분포 확인 → dataset.yaml 채우기

## 0. 환경 확인

In [1]:
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  GPU 미인식 — CUDA/드라이버 확인 필요')

CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 3070
VRAM: 8.6 GB


## 1. dataset.yaml 생성
> ⚠️ `names` 딕셔너리는 **EDA에서 확인한 실제 class_id**에 맞게 수정할 것

In [2]:
import yaml
from pathlib import Path

# 실제 데이터셋 절대 경로
DATASET_ROOT = Path(r'C:\work\python_area\AIotDataAnalysis\workspace_python\프로젝트2\dataset\track_a_images').resolve()

dataset_config = {
    'path': str(DATASET_ROOT).replace('\\', '/'),
    'train': 'images/train',
    'val':   'images/val',
    'nc': 8,
    'names': {
        0: 'scratch',      # 스크래치
        1: 'dent',         # 덴트
        2: 'paint_bubble', # 도장기포
        3: 'paint_drip',   # 도장흘림
        4: 'dust',         # 이물질
        5: 'orange_peel',  # 오렌지필
        6: 'crack',        # 크랙
        7: 'gap_fault',    # Gap불량
    }
}

yaml_path = Path('dataset.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(dataset_config, f, allow_unicode=True, sort_keys=False)

print('dataset.yaml 저장 완료:', yaml_path.resolve())
print(yaml_path.read_text(encoding='utf-8'))

dataset.yaml 저장 완료: C:\work\python_area\AIotDataAnalysis\workspace_python\프로젝트2\dataset.yaml
path: C:/work/python_area/AIotDataAnalysis/workspace_python/프로젝트2/dataset/track_a_images
train: images/train
val: images/val
nc: 8
names:
  0: scratch
  1: dent
  2: paint_bubble
  3: paint_drip
  4: dust
  5: orange_peel
  6: crack
  7: gap_fault



## 2. 학습 실행 — EXP-01 베이스라인 (YOLOv8s)

In [3]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

YAML_PATH = str(Path('dataset.yaml').resolve())
print('사용 yaml:', YAML_PATH)

# ── EXP-01: YOLOv8s 베이스라인 ──
model = YOLO('yolov8s.pt')

results = model.train(
    data=YAML_PATH,
    epochs=300,
    imgsz=640,
    batch=16,        # VRAM 부족 시 8로 변경
    seed=42,
    patience=50,
    project='runs/baseline',
    name='yolov8s_exp01',
    exist_ok=True,
    # 클래스 불균형 4.9:1 → 기본 증강으로 충분, cls_weights 미사용
)

print('\n학습 완료')
print('결과 저장 경로:', results.save_dir)

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
사용 yaml: C:\work\python_area\AIotDataAnalysis\workspace_python\프로젝트2\dataset.yaml
Ultralytics 8.4.34  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\work\python_area\AIotDataAnalysis\workspace_python\2\dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, ex

: 

## 3. 검증 (Val) 실행

In [ ]:
from pathlib import Path

YAML_PATH  = str(Path('dataset.yaml').resolve())
BEST_PT    = Path('runs/baseline/yolov8s_exp01/weights/best.pt')

if not BEST_PT.exists():
    print('best.pt 없음 — 학습 먼저 실행하세요')
else:
    best_model = YOLO(str(BEST_PT))
    val_results = best_model.val(data=YAML_PATH)

    print('\n=== EXP-01 평가 결과 ===')
    print(f'mAP@0.5:      {val_results.box.map50:.4f}')
    print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
    print(f'Precision:    {val_results.box.mp:.4f}')
    print(f'Recall:       {val_results.box.mr:.4f}')

## 4. 샘플 추론 시각화

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BEST_PT   = Path('runs/baseline/yolov8s_exp01/weights/best.pt')
VAL_DIR   = Path(r'C:\work\python_area\AIotDataAnalysis\workspace_python\프로젝트2\dataset\track_a_images\images\val')

val_imgs = list(VAL_DIR.glob('*.jpg'))
if not val_imgs:
    print('val 이미지 없음 — 경로 확인')
elif not BEST_PT.exists():
    print('best.pt 없음 — 학습 먼저 실행')
else:
    sample_img = str(val_imgs[0])
    pred = best_model.predict(
        source=sample_img, save=True,
        project='runs/predict', name='exp01_sample', exist_ok=True
    )

    result_img = Path(f'runs/predict/exp01_sample/{Path(sample_img).name}')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(mpimg.imread(sample_img))
    axes[0].set_title('원본')
    axes[0].axis('off')
    if result_img.exists():
        axes[1].imshow(mpimg.imread(str(result_img)))
        axes[1].set_title('EXP-01 탐지 결과')
    else:
        axes[1].set_title('결과 이미지 없음')
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('inference_exp01_sample.png', dpi=100)
    plt.show()
    print('저장:', 'inference_exp01_sample.png')